# EfficientMatch -- Expérience MixMatch (baseline)

Ce notebook implémente **MixMatch** (Berthelot et al., 2019) selon le protocole réduit du papier (budget $2^{17}$ itérations, schedule cosine recalé).

**Structure identique au notebook FlexMatch** (sections 1 à 5, 7 : imports, config, données, modèle, EMA/FLOPs, assemblage) pour permettre une comparaison directe et un diff minimal entre les deux fichiers. **Seule la section 6 (training step) diffère structurellement**, car MixMatch ne repose ni sur le seuillage ni sur la cohérence faible/forte, mais sur :
1. le \"guessing\" de pseudo-étiquettes par moyennage de $K$ augmentations faibles,
2. le \"sharpening\" de la distribution obtenue,
3. le Mixup entre l'ensemble concaténé (labellisé + non labellisé pseudo-étiqueté) et sa version mélangée.

Pas de superclasse : chaque étape de l'algorithme est visible explicitement dans `train_step_mixmatch`.

## 1. Imports

In [ ]:
import os
import time
import json
import math
import random

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset
from torch.utils.flop_counter import FlopCounterMode
import torchvision
import torchvision.transforms as transforms_v1
import torchvision.transforms.v2 as transforms_v2

print(f"Torch version: {torch.__version__}, CUDA disponible: {torch.cuda.is_available()}")

## 2. Configuration

Deux hyperparamètres supplémentaires spécifiques à MixMatch par rapport au notebook FlexMatch : `K_aug` (nombre d'augmentations faibles moyennées pour le guessing) et `sharpen_T` (température de sharpening). Pas de `tau` ici : MixMatch n'a pas de seuillage.

In [ ]:
CONFIG = {
    # --- Dataset ---
    "dataset": "cifar10",
    "data_root": "./data",
    "n_labels": 40,
    "num_classes": 10,

    # --- Hyperparamètres standards SSL ---
    "B": 64,
    "mu": 7,
    "lr": 0.03,
    "momentum": 0.9,
    "nesterov": True,
    "weight_decay": 5e-4,
    "ema_decay": 0.999,

    # --- Hyperparamètres spécifiques MixMatch ---
    "K_aug": 2,               # nombre d'augmentations faibles moyennées pour le pseudo-étiquetage
    "sharpen_T": 0.5,         # température de sharpening (papier original: 0.5)
    "alpha_mix": 0.75,        # paramètre de la loi Beta pour le Mixup
    "lambda_u_max": 75.0,     # poids max de la perte non supervisée (papier original CIFAR-10: 75)
    "rampup_length": 16384,   # nombre d'itérations pour le rampup linéaire de lambda_u

    # --- Budget d'entraînement (protocole réduit) ---
    "K": 2 ** 17,
    "iters_per_epoch": 1024,
    "eval_every": 512,
    "seed": 0,

    # --- OPTIMISATIONS DE VITESSE (débrayables) ---
    "use_amp": True,
    "cudnn_benchmark": True,
    "channels_last": True,
    "use_transforms_v2": True,  # True = torchvision.transforms.v2 (batch vectorisé) / False = v1 classique (par image)
    "num_workers": 4,
    "persistent_workers": True,
    "pin_memory": True,
    "debug_subset_size": None,
    "compile_model": False,

    # --- Divers ---
    "device": "cuda" if torch.cuda.is_available() else "cpu",
    "log_path": "./logs_mixmatch.json",
}

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(CONFIG["seed"])

if CONFIG["cudnn_benchmark"]:
    torch.backends.cudnn.benchmark = True

device = torch.device(CONFIG["device"])
print("Config chargée. Device:", device)

## 3. Traitement des données

MixMatch n'utilise qu'une seule famille d'augmentation, **faible** (pas de RandAugment/augmentation forte comme dans FixMatch/FlexMatch) : flip + translation, appliquée $K$ fois indépendamment sur chaque exemple non labellisé pour le guessing.

In [ ]:
CIFAR_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR_STD = (0.2471, 0.2435, 0.2616)

USE_TRANSFORMS_V2 = CONFIG["use_transforms_v2"]
T = transforms_v2 if USE_TRANSFORMS_V2 else transforms_v1

if USE_TRANSFORMS_V2:
    # v2 : un seul appel vectorisé sur tout le batch (CPU ou GPU), plus de boucle Python par image
    weak_transform = T.Compose([
        T.RandomHorizontalFlip(),
        T.RandomCrop(32, padding=4, padding_mode="reflect"),
        T.ToDtype(torch.float32, scale=True),
        T.Normalize(CIFAR_MEAN, CIFAR_STD),
    ])

    eval_transform = T.Compose([
        T.PILToTensor(),
        T.ToDtype(torch.float32, scale=True),
        T.Normalize(CIFAR_MEAN, CIFAR_STD),
    ])
else:
    # v1 (classique) : transform appliqué image par image (boucle Python) dans la boucle d'entraînement
    weak_transform = T.Compose([
        T.RandomHorizontalFlip(),
        T.RandomCrop(32, padding=4, padding_mode="reflect"),
        T.ToTensor(),
        T.Normalize(CIFAR_MEAN, CIFAR_STD),
    ])

    eval_transform = T.Compose([
        T.ToTensor(),
        T.Normalize(CIFAR_MEAN, CIFAR_STD),
    ])


def apply_batch(transform, raw_batch):
    """Bascule transparente v1 (liste de PIL, boucle Python) / v2 (batch tenseur vectorisé)."""
    if USE_TRANSFORMS_V2:
        return transform(raw_batch.to(device, non_blocking=True))
    return torch.stack([transform(img) for img in raw_batch]).to(device, non_blocking=True)


def ssl_collate(batch):
    """Collate custom : le DataLoader ne sait pas empiler nativement une liste d'images PIL (mode v1)."""
    imgs, labels = zip(*batch)
    imgs = torch.stack(imgs) if USE_TRANSFORMS_V2 else list(imgs)
    return imgs, torch.tensor(labels)


class SSLDataset(Dataset):
    def __init__(self, base_dataset, indices):
        self.base_dataset = base_dataset
        self.indices = indices

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        img, label = self.base_dataset[self.indices[idx]]
        if USE_TRANSFORMS_V2:
            img = transforms_v2.functional.pil_to_tensor(img)  # uint8 CHW -> collate direct en batch tenseur
        return img, label


def make_ssl_split(base_dataset, n_labels, num_classes, seed=0):
    rng = np.random.RandomState(seed)
    targets = np.array(base_dataset.targets)
    n_per_class = n_labels // num_classes
    labeled_idx = []
    for c in range(num_classes):
        idx_c = np.where(targets == c)[0]
        rng.shuffle(idx_c)
        labeled_idx.extend(idx_c[:n_per_class].tolist())
    labeled_idx = np.array(labeled_idx)
    unlabeled_idx = np.arange(len(base_dataset))
    return labeled_idx, unlabeled_idx


def load_datasets(cfg):
    if cfg["dataset"] == "cifar10":
        train_base = torchvision.datasets.CIFAR10(cfg["data_root"], train=True, download=True)
        test_base = torchvision.datasets.CIFAR10(cfg["data_root"], train=False, download=True, transform=eval_transform)
    elif cfg["dataset"] == "cifar100":
        train_base = torchvision.datasets.CIFAR100(cfg["data_root"], train=True, download=True)
        test_base = torchvision.datasets.CIFAR100(cfg["data_root"], train=False, download=True, transform=eval_transform)
    else:
        raise NotImplementedError(
            f"Dataset {cfg['dataset']} non branché ici -- ajouter le chargement MedMNIST (PathMNIST) via medmnist.PathMNIST"
        )

    labeled_idx, unlabeled_idx = make_ssl_split(train_base, cfg["n_labels"], cfg["num_classes"], seed=cfg["seed"])

    if cfg["debug_subset_size"] is not None:
        unlabeled_idx = unlabeled_idx[: cfg["debug_subset_size"]]
        test_base = Subset(test_base, list(range(min(len(test_base), cfg["debug_subset_size"]))))

    labeled_set = SSLDataset(train_base, labeled_idx)
    unlabeled_set = SSLDataset(train_base, unlabeled_idx)

    return labeled_set, unlabeled_set, test_base


def infinite_loader(dataset, batch_size, cfg, shuffle=True):
    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=cfg["num_workers"],
        pin_memory=cfg["pin_memory"],
        persistent_workers=cfg["persistent_workers"] and cfg["num_workers"] > 0,
        drop_last=True,
        collate_fn=ssl_collate,
    )
    while True:
        for batch in loader:
            yield batch

## 4. Modèle : WideResNet-28-2

Identique au notebook FlexMatch, pour garantir que l'architecture n'est pas une variable confondante entre les deux baselines.

In [ ]:
class BasicBlock(nn.Module):
    def __init__(self, in_planes, out_planes, stride, drop_rate=0.0):
        super().__init__()
        self.bn1 = nn.BatchNorm2d(in_planes)
        self.relu1 = nn.LeakyReLU(0.1, inplace=True)
        self.conv1 = nn.Conv2d(in_planes, out_planes, 3, stride=stride, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_planes)
        self.relu2 = nn.LeakyReLU(0.1, inplace=True)
        self.conv2 = nn.Conv2d(out_planes, out_planes, 3, stride=1, padding=1, bias=False)
        self.drop_rate = drop_rate
        self.equal_io = in_planes == out_planes and stride == 1
        self.shortcut = None if self.equal_io else nn.Conv2d(in_planes, out_planes, 1, stride=stride, bias=False)

    def forward(self, x):
        out = self.relu1(self.bn1(x))
        shortcut = x if self.equal_io else self.shortcut(out)
        out = self.conv1(out)
        out = self.relu2(self.bn2(out))
        if self.drop_rate > 0:
            out = F.dropout(out, p=self.drop_rate, training=self.training)
        out = self.conv2(out)
        return out + shortcut


class WideResNet(nn.Module):
    def __init__(self, num_classes=10, depth=28, widen_factor=2, drop_rate=0.0):
        super().__init__()
        n_channels = [16, 16 * widen_factor, 32 * widen_factor, 64 * widen_factor]
        assert (depth - 4) % 6 == 0
        n = (depth - 4) // 6

        self.conv1 = nn.Conv2d(3, n_channels[0], 3, stride=1, padding=1, bias=False)
        self.block1 = self._make_block(n_channels[0], n_channels[1], n, stride=1, drop_rate=drop_rate)
        self.block2 = self._make_block(n_channels[1], n_channels[2], n, stride=2, drop_rate=drop_rate)
        self.block3 = self._make_block(n_channels[2], n_channels[3], n, stride=2, drop_rate=drop_rate)
        self.bn1 = nn.BatchNorm2d(n_channels[3])
        self.relu = nn.LeakyReLU(0.1, inplace=True)
        self.fc = nn.Linear(n_channels[3], num_classes)
        self.n_channels = n_channels[3]

    def _make_block(self, in_planes, out_planes, num_layers, stride, drop_rate):
        layers = [BasicBlock(in_planes, out_planes, stride, drop_rate)]
        for _ in range(1, num_layers):
            layers.append(BasicBlock(out_planes, out_planes, 1, drop_rate))
        return nn.Sequential(*layers)

    def forward(self, x):
        out = self.conv1(x)
        out = self.block1(out)
        out = self.block2(out)
        out = self.block3(out)
        out = self.relu(self.bn1(out))
        out = F.adaptive_avg_pool2d(out, 1).flatten(1)
        return self.fc(out)


def build_model(cfg):
    model = WideResNet(num_classes=cfg["num_classes"], depth=28, widen_factor=2)
    model = model.to(device)
    if cfg["channels_last"]:
        model = model.to(memory_format=torch.channels_last)
    if cfg["compile_model"]:
        model = torch.compile(model)
    return model

## 5. EMA, FLOPs (mesure réelle) et évaluation

In [ ]:
class EMA:
    def __init__(self, model, decay):
        self.decay = decay
        self.shadow = {k: v.detach().clone() for k, v in model.state_dict().items()}

    @torch.no_grad()
    def update(self, model):
        for k, v in model.state_dict().items():
            if v.dtype.is_floating_point:
                self.shadow[k].mul_(self.decay).add_(v.detach(), alpha=1 - self.decay)
            else:
                self.shadow[k] = v.detach().clone()

    def copy_to(self, model):
        model.load_state_dict(self.shadow, strict=True)


def estimate_flops_per_iter(model, cfg):
    """Mesure réelle des FLOPs (forward + backward) pour une itération MixMatch, via FlopCounterMode.
    Exécutée UNE SEULE FOIS avant l'entraînement -- ne pas appeler pendant la boucle (surcoût de dispatch).

    Note importante par rapport au notebook FlexMatch : ici le nombre de forward passes du bloc
    non supervisé est déterminé par K_aug (guessing) + le Mixup (pas de vue \"forte\" séparée),
    donc le total diffère structurellement de FixMatch/FlexMatch -- c'est attendu et fait partie
    de la comparaison d'efficacité entre familles de méthodes (cf. papier, section Method).
    """
    device_ = next(model.parameters()).device
    model.train()

    B, muB = cfg["B"], cfg["mu"] * cfg["B"]
    dummy_x = torch.randn(B, 3, 32, 32, device=device_)
    dummy_u = torch.randn(muB, 3, 32, 32, device=device_)
    dummy_labels_x = torch.randint(0, cfg["num_classes"], (B,), device=device_)

    model.zero_grad(set_to_none=True)

    with FlopCounterMode(display=False) as flop_counter:
        # K_aug forward passes pour le guessing (sans grad, comme dans le vrai training step)
        with torch.no_grad():
            for _ in range(cfg["K_aug"]):
                _ = model(dummy_u)

        # forward sur les données mixées (labellisées + non labellisées), avec gradient
        logits_xp = model(dummy_x)
        logits_up = model(dummy_u)
        loss = F.cross_entropy(logits_xp, dummy_labels_x) + F.mse_loss(
            torch.softmax(logits_up, dim=-1), torch.softmax(logits_up.detach(), dim=-1)
        )
        loss.backward()

    model.zero_grad(set_to_none=True)
    return flop_counter.get_total_flops()


@torch.no_grad()
def evaluate(model, test_loader):
    model.eval()
    correct, total = 0, 0
    for imgs, labels in test_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        logits = model(imgs)
        preds = logits.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
    model.train()
    return correct / total

## 6. Boucle d'entraînement -- MixMatch

Chaque étape de l'algorithme original (Berthelot et al., 2019) est explicite :
1. Batch labellisé (une seule augmentation faible).
2. Batch non labellisé, **$K$ augmentations faibles indépendantes**.
3. **Guessing** : moyenne des $K$ prédictions.
4. **Sharpening** : aiguisage de la distribution moyenne avec température $T$.
5. **Concaténation + mélange (shuffle)** de $\mathcal{X}$ (labellisé) et $\hat{\mathcal{U}}$ (non labellisé pseudo-étiqueté, répété $K$ fois).
6. **MixUp** entre l'ensemble original et sa version mélangée.
7. Perte $\mathcal{L}_X$ (entropie croisée, partie labellisée) et $\mathcal{L}_U$ (MSE, partie non labellisée), combinées avec un poids $\lambda_u(t)$ en **rampup linéaire**.

In [ ]:
def sharpen(p, T):
    """Aiguise une distribution de probabilité : p_i^(1/T) / sum_j p_j^(1/T)."""
    p_sharp = p ** (1.0 / T)
    return p_sharp / p_sharp.sum(dim=-1, keepdim=True)


def mixup(x1, p1, x2, p2, alpha):
    """Mélange convexe entre deux paires (image, distribution de probabilité).
    lam est forcé >= 0.5 (convention MixMatch/MixUp) pour que le premier élément de la paire
    domine toujours le mélange -- important pour préserver l'ordre X'/U' après mixage.
    """
    lam = np.random.beta(alpha, alpha)
    lam = max(lam, 1 - lam)
    x = lam * x1 + (1 - lam) * x2
    p = lam * p1 + (1 - lam) * p2
    return x, p


def rampup_lambda_u(k, cfg):
    """Rampup linéaire de lambda_u de 0 à lambda_u_max sur rampup_length itérations.
    Nécessaire dans MixMatch (contrairement à FixMatch/FlexMatch) car en tout début
    d'entraînement les pseudo-étiquettes guessées sont très peu fiables -- cf. discussion
    papier, section Method (comparaison avec le filtrage par confiance d'EfficientMatch).
    """
    return cfg["lambda_u_max"] * min(1.0, k / cfg["rampup_length"])


def train_step_mixmatch(model, ema, optimizer, scaler, k, cfg, labeled_iter, unlabeled_iter):
    """Une itération d'entraînement MixMatch."""

    # --- Étape 1 : batch labellisé (une seule vue faible) ---
    imgs_x_raw, labels_x_int = next(labeled_iter)
    imgs_x = apply_batch(weak_transform, imgs_x_raw)
    labels_x_int = labels_x_int.to(device, non_blocking=True)
    labels_x = F.one_hot(labels_x_int, cfg["num_classes"]).float()

    # --- Étape 2 : batch non labellisé, K augmentations faibles indépendantes ---
    imgs_u_raw, _ = next(unlabeled_iter)
    imgs_u_augs = [apply_batch(weak_transform, imgs_u_raw) for _ in range(cfg["K_aug"])]

    if cfg["channels_last"]:
        imgs_x = imgs_x.to(memory_format=torch.channels_last)
        imgs_u_augs = [u.to(memory_format=torch.channels_last) for u in imgs_u_augs]

    optimizer.zero_grad(set_to_none=True)

    with torch.autocast(device_type=cfg["device"], enabled=cfg["use_amp"]):

        # --- Étape 3 : guessing -- moyenne des K prédictions (sans gradient) ---
        with torch.no_grad():
            probs_sum = torch.zeros(imgs_u_augs[0].size(0), cfg["num_classes"], device=device)
            for u_aug in imgs_u_augs:
                probs_sum += F.softmax(model(u_aug), dim=-1)
            probs_avg = probs_sum / cfg["K_aug"]

            # --- Étape 4 : sharpening ---
            guessed_labels = sharpen(probs_avg, cfg["sharpen_T"])
            # répété K fois pour être aligné avec les K vues augmentées dans la concaténation
            guessed_labels_rep = guessed_labels.repeat(cfg["K_aug"], 1)

        imgs_u_cat = torch.cat(imgs_u_augs, dim=0)  # (K_aug * mu*B, C, H, W)

        # --- Étape 5 : concaténation labellisé + non labellisé, puis mélange ---
        all_imgs = torch.cat([imgs_x, imgs_u_cat], dim=0)
        all_labels = torch.cat([labels_x, guessed_labels_rep], dim=0)
        perm = torch.randperm(all_imgs.size(0), device=device)
        all_imgs_shuffled = all_imgs[perm]
        all_labels_shuffled = all_labels[perm]

        # --- Étape 6 : MixUp entre l'ensemble original et sa version mélangée ---
        mixed_imgs, mixed_labels = mixup(
            all_imgs, all_labels, all_imgs_shuffled, all_labels_shuffled, cfg["alpha_mix"]
        )

        n_x = imgs_x.size(0)  # les n_x premiers éléments correspondent à la partie \"labellisée\" du mélange
        mixed_x, mixed_labels_x = mixed_imgs[:n_x], mixed_labels[:n_x]
        mixed_u, mixed_labels_u = mixed_imgs[n_x:], mixed_labels[n_x:]

        logits_mixed_x = model(mixed_x)
        logits_mixed_u = model(mixed_u)

        # --- Étape 7 : pertes ---
        loss_x = -(mixed_labels_x * F.log_softmax(logits_mixed_x, dim=-1)).sum(dim=-1).mean()
        loss_u = F.mse_loss(F.softmax(logits_mixed_u, dim=-1), mixed_labels_u)
        lambda_u = rampup_lambda_u(k, cfg)
        loss = loss_x + lambda_u * loss_u

    # --- Étape 8 : backward + optimisation ---
    if cfg["use_amp"]:
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
    else:
        loss.backward()
        optimizer.step()

    # --- Étape 9 : mise à jour EMA ---
    ema.update(model)

    return {
        "loss": loss.item(),
        "loss_x": loss_x.item(),
        "loss_u": loss_u.item(),
        "lambda_u": lambda_u,
    }

## 7. Assemblage : modèle, optimiseur, scheduler

In [ ]:
def cosine_schedule(optimizer, k, K):
    base_lr = optimizer.defaults["lr"]
    new_lr = base_lr * math.cos(7 * math.pi * k / (16 * K))
    for group in optimizer.param_groups:
        group["lr"] = max(new_lr, 0.0)


labeled_set, unlabeled_set, test_set = load_datasets(CONFIG)

labeled_iter = infinite_loader(labeled_set, CONFIG["B"], CONFIG, shuffle=True)
unlabeled_iter = infinite_loader(unlabeled_set, CONFIG["mu"] * CONFIG["B"], CONFIG, shuffle=True)
test_loader = DataLoader(test_set, batch_size=256, shuffle=False, num_workers=CONFIG["num_workers"])

model = build_model(CONFIG)
ema = EMA(model, CONFIG["ema_decay"])

optimizer = torch.optim.SGD(
    model.parameters(),
    lr=CONFIG["lr"],
    momentum=CONFIG["momentum"],
    nesterov=CONFIG["nesterov"],
    weight_decay=CONFIG["weight_decay"],
)

scaler = torch.amp.GradScaler(enabled=CONFIG["use_amp"])

# Mesure UNIQUE avant l'entraînement -- ne jamais appeler estimate_flops_per_iter pendant la boucle
flops_per_iter = estimate_flops_per_iter(model, CONFIG)
print(f"FLOPs (mesurés) par itération : {flops_per_iter:.3e}")
print(f"FLOPs totaux estimés sur tout l'entraînement : {flops_per_iter * CONFIG['K']:.3e}")
print(f"Budget total : {CONFIG['K']} itérations")

## 8. Boucle principale + logging

In [ ]:
logs = []
eval_model = build_model(CONFIG)

start_time = time.time()
model.train()

for k in range(1, CONFIG["K"] + 1):
    cosine_schedule(optimizer, k, CONFIG["K"])

    step_metrics = train_step_mixmatch(model, ema, optimizer, scaler, k, CONFIG,
                                        labeled_iter, unlabeled_iter)

    if k % CONFIG["eval_every"] == 0 or k == CONFIG["K"]:
        ema.copy_to(eval_model)
        acc = evaluate(eval_model, test_loader)
        elapsed = time.time() - start_time
        cumulative_flops = flops_per_iter * k  # valide : FLOPs/itération constants (pas de curriculum de taille)

        log_entry = {
            "iteration": k,
            "elapsed_seconds": elapsed,
            "cumulative_flops": cumulative_flops,
            "eval_accuracy": acc,
            **step_metrics,
        }
        logs.append(log_entry)
        print(f"[iter {k:>7}/{CONFIG['K']}] acc={acc:.4f} loss={step_metrics['loss']:.4f} "
              f"lambda_u={step_metrics['lambda_u']:.2f} elapsed={elapsed/60:.1f}min")

        with open(CONFIG["log_path"], "w") as f:
            json.dump({"config": CONFIG, "logs": logs}, f, indent=2)

print("Entraînement terminé.")

## 9. Notes de comparabilité avec FlexMatch/FixMatch

- Les sections 1 à 5 et 7 sont **volontairement identiques** au notebook FlexMatch (mêmes classes `WideResNet`, `EMA`, `SSLDataset`, mêmes hyperparamètres communs) : toute différence de résultat entre les deux runs provient donc uniquement de la section 6 (algorithme), et non d'une divergence incidente de pipeline.
- Le nombre de forward passes par itération diffère structurellement de FixMatch/FlexMatch (`K_aug` guessing passes + 2 passes sur les données mixées, contre 3 passes fixes chez FixMatch/FlexMatch) -- c'est une différence *de fond* entre les deux familles de méthodes, pas un artefact d'implémentation, et c'est précisément ce que la métrique FLOPs mesurée en section 5 doit capturer pour le papier.
- Pour brancher EfficientMatch : reprendre `train_step_flexmatch` (seuillage) et y insérer un canal de Mixup filtré inspiré des étapes 5-6 de ce notebook (concaténation + mixup), en pondérant par le masque de confiance -- cf. Algorithme EfficientMatch, section Method du papier.